In [ ]:
import json
import logging
import os
import subprocess
import sys
import warnings
from pathlib import Path

os.environ.setdefault("LOGURU_LEVEL", "INFO")
os.environ.setdefault("NEMO_LOG_LEVEL", "ERROR")
warnings.filterwarnings("ignore", message=r".*concurrency.*deprecated.*")
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

# Hindi Indic ASR Benchmark Tutorial

This notebook follows the paired-backend methodology used by the ALM tutorial while executing the exact `audio_indic_asr_xenna` and `audio_indic_asr_raydata` entries through `benchmarking/run.py`. Both runs use the same pinned 531.7738-hour Hindi cohort, model engines, processor graph, Ray resources, instrumentation, and correctness gates; only the executor and its configured timeout/runtime requirement differ.

The benchmark-equivalent environment is one 128-CPU, 8× H100 node with compatible TensorRT and TensorRT-LLM runtimes.

## Step 1: Prepare the pinned public cohort

From the repository root, run:

```bash
python benchmarking/data_prep/prepare_audio_indic_asr_data.py \
  --output-path /data/audio_indic_asr_parakeet_hindi_531h_35376a11 \
  --cache-dir /data/_hf_cache/audio_indic_asr
```

This pins `ketav/parakeet-hindi-asr` at revision `35376a112c4b79318eeaba0c0dd1b6f1a9bf0ea0` and validates all 216,169 clips.

In [ ]:
# Run this notebook from the Curator repository root and update these roots.
REPO_ROOT = Path.cwd().resolve()
DATASETS_ROOT = Path("/data")
MODEL_WEIGHTS_ROOT = Path("/models")
OUTPUT_ROOT = REPO_ROOT / "indic_asr_benchmark_output"

required_paths = [
    DATASETS_ROOT / "audio_indic_asr_parakeet_hindi_531h_35376a11/manifest.jsonl",
    MODEL_WEIGHTS_ROOT / "audio_indic_asr/indic_canary_trtllm/engine_bfloat16_64_new",
    MODEL_WEIGHTS_ROOT / "audio_indic_asr/parakeet_indic_trt/encoder_fp16_b1_o8_m16_f8_o800_m4001",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    msg = f"Update the tutorial paths; missing: {missing}"
    raise FileNotFoundError(msg)

## Step 2: Define the exact runner

The Hydra tutorial runner selects an exact entry from the canonical benchmark config. This includes the 500 GB Ray object store, timeout, GPU recorder, input normalization, four-way sharding, model/runtime preflight, full processor chain, harness `exec_time_s`, 1:1 output validation, and YAML requirements. Each backend gets a fresh session directory.

In [ ]:
ENTRY_BY_BACKEND = {"xenna": "audio_indic_asr_xenna", "ray_data": "audio_indic_asr_raydata"}


def run_backend(backend: str, run_number: int = 1) -> dict:
    entry_name = ENTRY_BY_BACKEND[backend]
    session_name = f"indic-asr-{backend}-{run_number:03d}"
    session_dir = OUTPUT_ROOT / session_name
    if session_dir.exists():
        msg = f"Use a fresh session name: {session_dir}"
        raise FileExistsError(msg)
    command = [
        sys.executable,
        str(REPO_ROOT / "tutorials/audio/indic_asr/main.py"),
        "--config-path",
        ".",
        "--config-name",
        "pipeline",
        f"datasets_path={DATASETS_ROOT}",
        f"model_weights_path={MODEL_WEIGHTS_ROOT}",
        f"results_path={OUTPUT_ROOT}",
        f"session_name={session_name}",
        f"backend={backend}",
    ]
    subprocess.run(command, cwd=REPO_ROOT, check=True)  # noqa: S603
    result_path = session_dir / entry_name / "results.json"
    return json.loads(result_path.read_text())

## Step 3: Run Xenna

On the standard EOS 8× H100 runner, the accepted harness-level `exec_time_s` range is 600–900 seconds. The canonical entry evaluates this requirement.

In [ ]:
xenna_result = run_backend("xenna")
xenna_result

## Step 4: Run Ray Data

Ray Data is an ungated same-data comparison. Wait for the Xenna process and its Ray cluster to stop before starting it.

In [ ]:
ray_data_result = run_backend("ray_data")
ray_data_result

## Step 5: Compare the paired evidence

Cardinality, duration, prediction coverage, provenance, and WER coverage should satisfy the same correctness contract for both executors. Only Xenna has the 10–15 minute EOS timing criterion.

In [ ]:
metric_keys = [
    "pipeline_time_s",
    "num_output_rows",
    "total_audio_duration_hours",
    "input_output_coverage_ratio",
    "best_prediction_coverage_ratio",
    "wer_output_coverage_ratio",
    "mean_wer_pct",
]
comparison = {
    "exec_time_s": {"xenna": xenna_result["exec_time_s"], "ray_data": ray_data_result["exec_time_s"]},
    **{
        key: {"xenna": xenna_result["metrics"][key], "ray_data": ray_data_result["metrics"][key]}
        for key in metric_keys
    },
}
comparison